Inicio de dim organization con su condicional para sumar varios registros

In [0]:
%sql
-- Raw extraction with strict deduplication and 9xxx synthetic records
TRUNCATE TABLE workspace.silver.dim_organization_tmp;

INSERT INTO workspace.silver.dim_organization_tmp (
    SalesOrganization,
    DistributionChannel,
    OrganizationDivision,
    _rescued_data,
    _ingestion_timestamp,
    Status_Cleansing,
    Status_DQ1,
    Status_DQ2,
    Status_DQ3,
    Status_Process
)
-- Deduplicated real organizations (keeps latest timestamp)
SELECT 
    SalesOrganization,
    DistributionChannel,
    OrganizationDivision,
    _rescued_data,
    _ingestion_timestamp,
    'PENDING' AS Status_Cleansing,
    'PENDING' AS Status_DQ1,
    'PENDING' AS Status_DQ2,
    'PENDING' AS Status_DQ3,
    'IN_PROGRESS' AS Status_Process
FROM workspace.bronze.sales_order_header
WHERE SalesOrganization IS NOT NULL
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY SalesOrganization, DistributionChannel, OrganizationDivision 
    ORDER BY _ingestion_timestamp DESC NULLS LAST
) = 1

UNION ALL

-- Synthetic organizations for Gold analytics
SELECT 
    '9000' AS SalesOrganization,
    '90' AS DistributionChannel,
    '09' AS OrganizationDivision,
    NULL AS _rescued_data,
    current_timestamp() AS _ingestion_timestamp,
    'PENDING', 'PENDING', 'PENDING', 'PENDING', 'IN_PROGRESS'

UNION ALL

SELECT 
    '9000' AS SalesOrganization,
    '91' AS DistributionChannel,
    '09' AS OrganizationDivision,
    NULL AS _rescued_data,
    current_timestamp() AS _ingestion_timestamp,
    'PENDING', 'PENDING', 'PENDING', 'PENDING', 'IN_PROGRESS';

In [0]:
%sql
UPDATE workspace.silver.dim_organization_tmp
SET 
    SalesOrganization = trim(upper(SalesOrganization)),
    DistributionChannel = trim(upper(DistributionChannel)),
    OrganizationDivision = trim(upper(OrganizationDivision)),
    Status_Cleansing = 'COMPLETED'
WHERE Status_Cleansing = 'PENDING' 
  AND Status_Process = 'IN_PROGRESS';

In [0]:
%sql
UPDATE workspace.silver.dim_organization_tmp
SET 
    Status_DQ1 = CASE 
        WHEN SalesOrganization IS NULL OR SalesOrganization = ''
          OR DistributionChannel IS NULL OR DistributionChannel = ''
          OR OrganizationDivision IS NULL OR OrganizationDivision = '' 
        THEN 'FAILED'
        ELSE 'PASSED'
    END,
    Status_Process = CASE 
        WHEN SalesOrganization IS NULL OR SalesOrganization = ''
          OR DistributionChannel IS NULL OR DistributionChannel = ''
          OR OrganizationDivision IS NULL OR OrganizationDivision = '' 
        THEN 'QUARANTINED'
        ELSE Status_Process
    END
WHERE Status_Cleansing = 'COMPLETED' 
  AND Status_Process = 'IN_PROGRESS';

In [0]:
%sql
UPDATE workspace.silver.dim_organization_tmp
SET 
    Status_DQ2 = CASE 
        WHEN length(SalesOrganization) > 4
          OR length(DistributionChannel) > 2
          OR length(OrganizationDivision) > 2
        THEN 'FAILED'
        ELSE 'PASSED'
    END,
    Status_Process = CASE 
        WHEN length(SalesOrganization) > 4
          OR length(DistributionChannel) > 2
          OR length(OrganizationDivision) > 2
        THEN 'QUARANTINED'
        ELSE Status_Process
    END
WHERE Status_Process = 'IN_PROGRESS' 
  AND Status_DQ1 = 'PASSED';

In [0]:
%sql
UPDATE workspace.silver.dim_organization_tmp
SET 
    Status_DQ3 = CASE 
        WHEN _rescued_data IS NOT NULL THEN 'FAILED'
        ELSE 'PASSED'
    END,
    Status_Process = CASE 
        WHEN _rescued_data IS NOT NULL THEN 'QUARANTINED'
        ELSE Status_Process
    END
WHERE Status_Process = 'IN_PROGRESS' 
  AND Status_DQ2 = 'PASSED';

In [0]:
%sql
-- Sweeper: Consolidated routing of rejected records to Quarantine
INSERT INTO workspace.silver.sales_order_quarantine (
    SourceEntity,
    RecordIdentifier,
    FailedRule,
    FailureSeverity,
    RawRecord,
    IngestionTimestamp
)
SELECT 
    'DIM_ORGANIZATION' AS SourceEntity,
    concat(coalesce(SalesOrganization, 'NULL'), '-', coalesce(DistributionChannel, 'NULL'), '-', coalesce(OrganizationDivision, 'NULL')) AS RecordIdentifier,
    CASE 
        WHEN Status_DQ1 = 'FAILED' THEN 'DQ1_ORGANIZATION_KEYS_NULL'
        WHEN Status_DQ2 = 'FAILED' THEN 'DQ2_ORGANIZATION_LENGTH_EXCEEDED'
        WHEN Status_DQ3 = 'FAILED' THEN 'DQ3_RESCUED_DATA_CORRUPT'
        ELSE 'UNKNOWN_FAILURE'
    END AS FailedRule,
    'HARD' AS FailureSeverity,
    to_json(named_struct(
        'SalesOrganization', SalesOrganization,
        'DistributionChannel', DistributionChannel,
        'OrganizationDivision', OrganizationDivision,
        '_rescued_data', _rescued_data
    )) AS RawRecord,
    current_timestamp() AS IngestionTimestamp
FROM workspace.silver.dim_organization_tmp
WHERE Status_Process = 'QUARANTINED';

In [0]:
%sql
UPDATE workspace.silver.dim_organization_tmp
SET Status_Process = 'READY_FOR_STG'
WHERE Status_Process = 'IN_PROGRESS'
  AND Status_DQ1 = 'PASSED'
  AND Status_DQ2 = 'PASSED'
  AND Status_DQ3 = 'PASSED';

In [0]:
%sql
-- Promotion: Load and idempotency into target dimensión
MERGE INTO workspace.silver.dim_organization_stg AS tgt
USING (
    SELECT 
        SalesOrganization,
        DistributionChannel,
        OrganizationDivision,
        max(coalesce(_ingestion_timestamp, current_timestamp())) AS ValidFrom
    FROM workspace.silver.dim_organization_tmp
    WHERE Status_Process = 'READY_FOR_STG'
    GROUP BY 
        SalesOrganization,
        DistributionChannel,
        OrganizationDivision
) AS src
ON  tgt.SalesOrganization   = src.SalesOrganization
AND tgt.DistributionChannel  = src.DistributionChannel
AND tgt.OrganizationDivision = src.OrganizationDivision
AND tgt.IsCurrent = TRUE

WHEN NOT MATCHED THEN
    INSERT (
        SalesOrganization,
        DistributionChannel,
        OrganizationDivision,
        ValidFrom,
        ValidTo,
        IsCurrent
    )
    VALUES (
        src.SalesOrganization,
        src.DistributionChannel,
        src.OrganizationDivision,
        src.ValidFrom,
        NULL,
        TRUE
    );